In [1]:
import logging
import time
import warnings

warnings.filterwarnings("ignore")
for name in ["azure", "azure.ai.ml", "azure.identity", "urllib3", "mlflow", "opentelemetry"]:
    logging.getLogger(name).setLevel(logging.ERROR)

import json
import os
import requests
import pandas as pd
from datetime import datetime

from azure.identity import DefaultAzureCredential
from azure.ai.ml import MLClient, command, Input, Output

ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Workspace :", ml_client.workspace_name)

PRENOM = "arthur"
cluster_name = f"cc{PRENOM}"


def attendre(job):
    """Affiche le statut du job toutes les 30 secondes jusqu'à la fin."""
    while True:
        statut = ml_client.jobs.get(job.name).status
        print(time.strftime("%H:%M:%S"), job.display_name, "-", statut)
        if statut in ("Completed", "Failed", "Canceled"):
            return statut
        time.sleep(30)

Found the config file in: /config.json


Workspace : mlw-iwid-arthur


In [ ]:
df = pd.read_csv("students_dropout.csv", sep=";")
df.columns = df.columns.str.strip()

print("Lignes x colonnes :", df.shape)
print("Colonne cible d'origine : Target")
print("\nRépartition de Target :")
print(df["Target"].value_counts())

abandon = (df["Target"] == "Dropout").astype(int)
print("\nRépartition de la cible binaire 'abandon' :")
print(abandon.value_counts())
print(abandon.value_counts(normalize=True).round(3))

print("\nValeurs manquantes au total :", int(df.isna().sum().sum()))

Lignes x colonnes : (4424, 37)
Colonne cible d'origine : Target

Répartition de Target :
Graduate    2209
Dropout     1421
Enrolled     794
Name: Target, dtype: int64

Répartition de la cible binaire 'abandon' :
0    3003
1    1421
Name: Target, dtype: int64
0    0.679
1    0.321
Name: Target, dtype: float64

Valeurs manquantes au total : 0


In [5]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

data = Data(
    path="students_dropout.csv",
    type=AssetTypes.URI_FILE,
    name="students-data",
    version="1",
    description="Predict Students' Dropout and Academic Success (UCI), 4424 étudiants",
)
data = ml_client.data.create_or_update(data)
print("Data asset :", data.name, "version", data.version)

Uploading students_dropout.csv (< 1 MB): 100%|██████████| 533k/533k [00:00<00:00, 18.3MB/s]




Data asset : students-data version 1


In [6]:
os.makedirs("src", exist_ok=True)

In [7]:
%%writefile src/train.py
"""
train.py - Entraînement du modèle de prédiction d'abandon des étudiants (sujet F)
MLflow (paramètres, métriques, modèle) + validation et enregistrement conditionnel.
"""
import argparse

import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

TARGET = "abandon"


def parse_args():
    """1. Lecture des paramètres passés au script."""
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True,
                        help="Chemin du fichier CSV")
    parser.add_argument("--class_weight", type=str, default="none",
                        choices=["none", "balanced"],
                        help="Pondération des classes")
    parser.add_argument("--test_size", type=float, default=0.3,
                        help="Part des données gardée pour le test (0.3 = 30 %%)")
    parser.add_argument("--min_recall", type=float, default=0.5,
                        help="Recall minimum pour enregistrer le modèle")
    parser.add_argument("--model_name", type=str, default="students-model",
                        help="Nom du modèle dans le registre")
    return parser.parse_args()


def load_data(path):
    """2. Lecture (séparateur point-virgule) et préparation du sujet F."""
    df = pd.read_csv(path, sep=";")
    df.columns = df.columns.str.strip()
    df[TARGET] = (df["Target"] == "Dropout").astype(int)
    df = df.drop(columns=["Target"])
    return df


def train_model(X_train, y_train, class_weight):
    """3. Entraînement : standardisation + régression logistique."""
    cw = None if class_weight == "none" else class_weight
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight=cw, max_iter=1000),
    )
    model.fit(X_train, y_train)
    return model


def evaluate(model, X_test, y_test):
    """4. Évaluation sur le jeu de test."""
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "auc": roc_auc_score(y_test, y_proba),
    }


def main():
    args = parse_args()
    print(f"Paramètres : class_weight={args.class_weight}, "
          f"test_size={args.test_size}")

    df = load_data(args.data)
    print(f"Données : {df.shape[0]} lignes, {df.shape[1]} colonnes, "
          f"taux d'abandon = {df[TARGET].mean():.1%}")
    X = df.drop(columns=[TARGET]).astype(float)
    y = df[TARGET]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=args.test_size, random_state=42, stratify=y)

    with mlflow.start_run():
        mlflow.log_param("class_weight", args.class_weight)
        mlflow.log_param("test_size", args.test_size)

        model = train_model(X_train, y_train, args.class_weight)
        metrics = evaluate(model, X_test, y_test)
        for name, value in metrics.items():
            print(name, ":", round(value, 4))

        mlflow.log_metrics(metrics)

        signature = infer_signature(X_test, model.predict(X_test))
        mlflow.sklearn.log_model(model, artifact_path="model", signature=signature)
        print("Modèle enregistré dans le run MLflow (dossier model/)")

        if metrics["recall"] >= args.min_recall:
            run_id = mlflow.active_run().info.run_id
            mlflow.register_model(f"runs:/{run_id}/model", args.model_name)
            mlflow.set_tag("validation", "acceptée")
            print(f"Modèle validé (recall >= {args.min_recall}) : "
                  f"nouvelle version de {args.model_name}")
        else:
            mlflow.set_tag("validation", "refusée")
            print(f"Modèle refusé : recall {metrics['recall']:.3f} "
                  f"< {args.min_recall}")


if __name__ == "__main__":
    main()

Overwriting src/train.py


In [8]:
def lancer_job(class_weight, test_size, display_name, min_recall=1.1):
    """Soumet un job d'entraînement.
    min_recall=1.1 est impossible à atteindre : aux étapes 3 et 4 on ne veut rien
    enregistrer, c'est le rôle du pipeline (étape 5). Le tag 'refusée' est normal ici."""
    job = command(
        code="./src",
        command="python train.py --data ${{inputs.data}} "
                "--class_weight ${{inputs.class_weight}} "
                "--test_size ${{inputs.test_size}} "
                "--min_recall ${{inputs.min_recall}}",
        inputs={
            "data": Input(type="uri_file", path="azureml:students-data:1"),
            "class_weight": class_weight,
            "test_size": test_size,
            "min_recall": min_recall,
        },
        environment="env-credit-sklearn:1",
        compute=cluster_name,
        experiment_name="students-dropout",
        display_name=display_name,
    )
    return ml_client.jobs.create_or_update(job)

In [9]:
job1 = lancer_job("none", 0.3, "students-none-30")
print(job1.studio_url)
attendre(job1)

Uploading src (0.0 MBs): 100%|██████████| 4604/4604 [00:00<00:00, 88688.02it/s]




https://ml.azure.com/runs/dreamy_cheese_kblqpkf9ln?wsid=/subscriptions/4da8365b-d320-4158-a985-02745767778d/resourcegroups/rg-iwid-arthur/workspaces/mlw-iwid-arthur&tid=88eebcae-d6e6-4ef7-bba4-4c34f4c2d5e0
13:34:51 students-none-30 - JobStatus.STARTING
13:35:21 students-none-30 - JobStatus.QUEUED
13:35:52 students-none-30 - JobStatus.QUEUED
13:36:22 students-none-30 - JobStatus.QUEUED
13:36:52 students-none-30 - JobStatus.QUEUED
13:37:23 students-none-30 - JobStatus.QUEUED
13:37:53 students-none-30 - JobStatus.QUEUED
13:38:23 students-none-30 - JobStatus.RUNNING
13:38:53 students-none-30 - JobStatus.RUNNING
13:39:24 students-none-30 - JobStatus.RUNNING
13:39:54 students-none-30 - JobStatus.RUNNING
13:40:24 students-none-30 - JobStatus.COMPLETED


<JobStatus.COMPLETED: 'Completed'>

In [10]:
job2 = lancer_job("balanced", 0.3, "students-balanced-30")
job3 = lancer_job("balanced", 0.2, "students-balanced-20")

attendre(job2)
attendre(job3)

13:40:50 students-balanced-30 - JobStatus.QUEUED
13:41:20 students-balanced-30 - JobStatus.RUNNING
13:41:50 students-balanced-30 - JobStatus.COMPLETED
13:41:51 students-balanced-20 - JobStatus.RUNNING
13:42:21 students-balanced-20 - JobStatus.COMPLETED


<JobStatus.COMPLETED: 'Completed'>

In [11]:
import mlflow

ws = ml_client.workspaces.get(ml_client.workspace_name)
mlflow.set_tracking_uri(ws.mlflow_tracking_uri)

runs = mlflow.search_runs(experiment_names=["students-dropout"])
runs = runs.dropna(subset=["metrics.recall"])

colonnes = ["run_id", "params.class_weight", "params.test_size",
            "metrics.accuracy", "metrics.precision", "metrics.recall",
            "metrics.f1", "metrics.auc"]
runs = runs[colonnes].sort_values("metrics.recall", ascending=False)
runs

,run_id,params.class_weight,params.test_size,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1,metrics.auc
1,olden_lock_zffxr0vwfs,balanced,0.3,0.874247,0.782609,0.843091,0.811725,0.930982
2,lemon_knee_fpy3bdrcdt,balanced,0.2,0.873446,0.786667,0.830986,0.808219,0.927175
0,dreamy_cheese_kblqpkf9ln,none,0.3,0.888554,0.882192,0.754098,0.813131,0.930468


In [12]:
os.makedirs("pipeline", exist_ok=True)

In [ ]:
%%writefile pipeline/prep.py
"""
prep.py - Étape 1 du pipeline : préparation des données (sujet F)
Lit le fichier (séparateur point-virgule), supprime les lignes incomplètes
et les doublons, puis écrit un fichier propre. La cible est créée dans train.py.
"""
import argparse
import os

import pandas as pd


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="CSV d'entrée")
    parser.add_argument("--output", type=str, required=True, help="Dossier de sortie")
    args = parser.parse_args()

    df = pd.read_csv(args.data, sep=";")
    df.columns = df.columns.str.strip()
    print("Lignes en entrée :", len(df))

    df = df.dropna()
    df = df.drop_duplicates()
    print("Lignes en sortie :", len(df))

    os.makedirs(args.output, exist_ok=True)
    df.to_csv(os.path.join(args.output, "clean.csv"), sep=";", index=False)
    print("Fichier écrit :", os.path.join(args.output, "clean.csv"))


if __name__ == "__main__":
    main()

Overwriting pipeline/prep.py


In [14]:
prep_step = command(
    name="prep_students",
    display_name="1 - Préparation des données",
    code="./pipeline",
    command="python prep.py --data ${{inputs.data}} --output ${{outputs.clean_data}}",
    inputs={"data": Input(type="uri_file")},
    outputs={"clean_data": Output(type="uri_folder")},
    environment="env-credit-sklearn:1",
)

train_step = command(
    name="train_students",
    display_name="2 - Entraînement et validation",
    code="./src",
    command="python train.py --data ${{inputs.data}}/clean.csv "
            "--class_weight ${{inputs.class_weight}} "
            "--test_size ${{inputs.test_size}} "
            "--min_recall ${{inputs.min_recall}}",
    inputs={
        "data": Input(type="uri_folder"),
        "class_weight": Input(type="string"),
        "test_size": Input(type="number"),
        "min_recall": Input(type="number"),
    },
    environment="env-credit-sklearn:1",
)

In [15]:
from azure.ai.ml.dsl import pipeline


@pipeline(description="Préparation, entraînement et validation du modèle students")
def students_pipeline(raw_data, class_weight, test_size, min_recall):
    prep_job = prep_step(data=raw_data)
    train_job = train_step(
        data=prep_job.outputs.clean_data,
        class_weight=class_weight,
        test_size=test_size,
        min_recall=min_recall,
    )

In [16]:
SEUIL_RECALL = 0.7

pipeline_job = students_pipeline(
    raw_data=Input(type="uri_file", path="azureml:students-data:1"),
    class_weight="balanced",
    test_size=0.3,
    min_recall=SEUIL_RECALL,
)
pipeline_job.settings.default_compute = cluster_name
pipeline_job.display_name = "pipeline-balanced"

pipeline_job = ml_client.jobs.create_or_update(
    pipeline_job, experiment_name="students-pipeline")
print(pipeline_job.studio_url)
attendre(pipeline_job)

Uploading pipeline (0.0 MBs): 100%|██████████| 1333/1333 [00:00<00:00, 31817.89it/s]




https://ml.azure.com/runs/red_chicken_xkn0xq504m?wsid=/subscriptions/4da8365b-d320-4158-a985-02745767778d/resourcegroups/rg-iwid-arthur/workspaces/mlw-iwid-arthur&tid=88eebcae-d6e6-4ef7-bba4-4c34f4c2d5e0
13:46:07 pipeline-balanced - JobStatus.NOT_STARTED
13:46:37 pipeline-balanced - JobStatus.RUNNING
13:47:07 pipeline-balanced - JobStatus.RUNNING
13:47:38 pipeline-balanced - JobStatus.RUNNING
13:48:08 pipeline-balanced - JobStatus.RUNNING
13:48:38 pipeline-balanced - JobStatus.RUNNING
13:49:09 pipeline-balanced - JobStatus.RUNNING
13:49:39 pipeline-balanced - JobStatus.RUNNING
13:50:09 pipeline-balanced - JobStatus.RUNNING
13:50:39 pipeline-balanced - JobStatus.RUNNING
13:51:10 pipeline-balanced - JobStatus.RUNNING
13:51:40 pipeline-balanced - JobStatus.RUNNING
13:52:10 pipeline-balanced - JobStatus.RUNNING
13:52:41 pipeline-balanced - JobStatus.RUNNING
13:53:11 pipeline-balanced - JobStatus.RUNNING
13:53:41 pipeline-balanced - JobStatus.COMPLETED


<JobStatus.COMPLETED: 'Completed'>

In [17]:
pipeline_job2 = students_pipeline(
    raw_data=Input(type="uri_file", path="azureml:students-data:1"),
    class_weight="none",
    test_size=0.3,
    min_recall=SEUIL_RECALL,
)
pipeline_job2.settings.default_compute = cluster_name
pipeline_job2.display_name = "pipeline-none"

pipeline_job2 = ml_client.jobs.create_or_update(
    pipeline_job2, experiment_name="students-pipeline")
print(pipeline_job2.studio_url)
attendre(pipeline_job2)

https://ml.azure.com/runs/witty_receipt_w4vf0zpdsj?wsid=/subscriptions/4da8365b-d320-4158-a985-02745767778d/resourcegroups/rg-iwid-arthur/workspaces/mlw-iwid-arthur&tid=88eebcae-d6e6-4ef7-bba4-4c34f4c2d5e0
13:56:56 pipeline-none - JobStatus.NOT_STARTED
13:57:26 pipeline-none - JobStatus.RUNNING
13:57:57 pipeline-none - JobStatus.RUNNING
13:58:27 pipeline-none - JobStatus.RUNNING
13:58:57 pipeline-none - JobStatus.RUNNING
13:59:28 pipeline-none - JobStatus.RUNNING
13:59:58 pipeline-none - JobStatus.RUNNING
14:00:28 pipeline-none - JobStatus.RUNNING
14:00:59 pipeline-none - JobStatus.RUNNING
14:01:29 pipeline-none - JobStatus.RUNNING
14:01:59 pipeline-none - JobStatus.RUNNING
14:02:29 pipeline-none - JobStatus.RUNNING
14:03:00 pipeline-none - JobStatus.RUNNING
14:03:30 pipeline-none - JobStatus.RUNNING
14:04:01 pipeline-none - JobStatus.RUNNING
14:04:31 pipeline-none - JobStatus.RUNNING
14:05:01 pipeline-none - JobStatus.RUNNING
14:05:31 pipeline-none - JobStatus.RUNNING
14:06:02 pipelin

<JobStatus.COMPLETED: 'Completed'>

In [ ]:
for m in ml_client.models.list(name="students-model"):
    print(m.name, "- version", m.version)

students-model - version 2
students-model - version 1


In [2]:
from azure.ai.ml.entities import (ManagedOnlineEndpoint, ManagedOnlineDeployment,
                                  Environment, CodeConfiguration)

modele = ml_client.models.get(name="students-model", label="latest")
print("Modèle retenu :", modele.name, "version", modele.version)

Modèle retenu : students-model version 2


In [ ]:
data_asset = ml_client.data.get("students-data", version="1")
df = pd.read_csv(data_asset.path, sep=";")
df.columns = df.columns.str.strip()

exemples = df.drop(columns=["Target"]).head(3)
requete = {"data": exemples.to_dict(orient="records")}

with open("sample-request.json", "w") as f:
    json.dump(requete, f, indent=2)

print("Abandon réel des 3 étudiants :",
	(df["Target"] == "Dropout").astype(int).head(3).tolist())

Abandon réel des 3 étudiants : [1, 0, 1]


In [21]:
os.makedirs("deploy", exist_ok=True)

In [ ]:
%%writefile deploy/score.py
import os
import json
import pandas as pd
import mlflow.sklearn

model = None


def init():
    """Chargement du modèle au démarrage du conteneur."""
    global model
    racine = os.getenv("AZUREML_MODEL_DIR")

    chemin_modele = None
    for dossier, _, fichiers in os.walk(racine):
        if "MLmodel" in fichiers:
            chemin_modele = dossier
            break

    model = mlflow.sklearn.load_model(chemin_modele)
    print(f"[init] Modèle chargé depuis : {chemin_modele}")


def run(raw_data):
    """Appelée à chaque requête. raw_data est le texte JSON envoyé à l'API."""
    try:
        donnees = json.loads(raw_data)["data"]
        X = pd.DataFrame(donnees).astype(float)

        colonnes = getattr(model, "feature_names_in_", None)
        if colonnes is not None:
            manquantes = [c for c in colonnes if c not in X.columns]
            if manquantes:
                print(f"[run] ERREUR : colonnes manquantes {manquantes}")
                return {"erreur": f"{len(manquantes)} colonne(s) manquante(s) : {manquantes}"}
            X = X[list(colonnes)]

        predictions = model.predict(X).tolist()
        probabilites = model.predict_proba(X)[:, 1].round(3).tolist()

        print(f"[run] {len(X)} ligne(s) reçue(s), prédictions : {predictions}")
        return {"predictions": predictions, "probabilites_abandon": probabilites}

    except Exception as erreur:
        print(f"[run] ERREUR : {erreur}")
        return {"erreur": str(erreur)}

Overwriting deploy/score.py


In [23]:
%%writefile deploy/conda-deploy.yml
name: env-credit-deploy
channels:
  - conda-forge
dependencies:
  - python=3.10
  - pip
  - pip:
      - scikit-learn==1.5.2
      - pandas==2.2.3
      - numpy==1.26.4
      - joblib==1.4.2
      - mlflow==2.22.0
      - azureml-inference-server-http==1.4.1

Overwriting deploy/conda-deploy.yml


In [4]:
env_deploy = Environment(
    name="env-credit-deploy",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:latest",
    conda_file="deploy/conda-deploy.yml",
)
env_deploy = ml_client.environments.create_or_update(env_deploy)
print(env_deploy.name, "version", env_deploy.version)

env-credit-deploy version 1


In [ ]:
prenom = "arthur"
nom_endpoint = f"students-ep-arthur-louca-{datetime.now():%H%M}"
print("Nom de l'endpoint :", nom_endpoint)

endpoint = ManagedOnlineEndpoint(name=nom_endpoint, auth_mode="key",
                                description="API de prédiction d'abandon étudiant")
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

Nom de l'endpoint : students-ep-arthur-louca-1904


ManagedOnlineEndpoint({'public_network_access': <PublicNetworkAccessType.ENABLED: 'Enabled'>, 'provisioning_state': <EndpointProvisioningState.SUCCEEDED: 'Succeeded'>, 'scoring_uri': 'https://students-ep-arthur-louca-1904.germanywestcentral.inference.ml.azure.com/score', 'openapi_uri': 'https://students-ep-arthur-louca-1904.germanywestcentral.inference.ml.azure.com/swagger.json', 'name': 'students-ep-arthur-louca-1904', 'description': "API de prédiction d'abandon étudiant", 'tags': {}, 'properties': {'createdBy': 'Arthur GRAMONT', 'createdAt': '2026-11-04T19:04:08.191177+0000', 'lastModifiedAt': '2026-11-04T19:04:57.081522+0000', 'azureml.onlineendpointid': '/subscriptions/4da8365b-d320-4158-a985-02745767778d/resourcegroups/rg-iwid-arthur/providers/microsoft.machinelearningservices/workspaces/mlw-iwid-arthur/onlineendpoints/students-ep-arthur-louca-1904', 'AzureAsyncOperationUri': 'https://management.azure.com/subscriptions/4da8365b-d320-4158-a985-02745767778d/providers/Microsoft.Machi

In [6]:
deploiement = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=nom_endpoint,
    model=modele,
    environment=env_deploy,
    code_configuration=CodeConfiguration(code="deploy", scoring_script="score.py"),
    instance_type="Standard_F2s_v2",
    instance_count=1,
)
ml_client.online_deployments.begin_create_or_update(deploiement)

Check: endpoint students-ep-arthur-louca-1904 exists


........

In [7]:
from azure.core.exceptions import ResourceNotFoundError

def attendre_deploiement(nom, endpoint):
    """Affiche l'état du déploiement toutes les 30 secondes jusqu'à la fin."""
    while True:
        try:
            d = ml_client.online_deployments.get(name=nom, endpoint_name=endpoint)
            print(f"{datetime.now():%H:%M:%S} - {d.provisioning_state}")
            if d.provisioning_state in ("Succeeded", "Failed", "Canceled"):
                return d.provisioning_state
        except ResourceNotFoundError:
            print("Déploiement pas encore visible, on attend...")
        time.sleep(30)

etat = attendre_deploiement("blue", nom_endpoint)

19:05:57 - Creating
.....19:06:27 - Creating
......19:06:57 - Creating
......19:07:27 - Creating
......19:07:57 - Creating
......19:08:28 - Creating
.....19:08:58 - Creating
......19:09:28 - Creating
......19:09:58 - Creating
.....19:10:29 - Creating
......19:10:59 - Creating
......19:11:29 - Creating
.....19:11:59 - Creating
......19:12:29 - Creating
......19:13:00 - Creating
......19:13:30 - Creating
.....19:14:00 - Creating
......19:14:30 - Creating
......19:15:01 - Succeeded
.

In [28]:
print(ml_client.online_deployments.get_logs(name="blue",
                                            endpoint_name=nom_endpoint, lines=40))

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: Succeeded

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-11-04T14:19:03.181652Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-11-04T14:19:04.073472Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-11-04T14:19:56.978262Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-11-04T14:19:56.978262Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-11-04T14:19:57.02587Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-11-04T14:19:57.075828Z, Message: Started container inference-server
Kind: Pod, Name: ContainerReady, Type: Normal, Time: 2026-11-04T14:20:18.878644342Z, Message: Container is ready

Container logs:
2026-11-04 14:

In [8]:
endpoint = ml_client.online_endpoints.get(nom_endpoint)
endpoint.traffic = {"blue": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

endpoint = ml_client.online_endpoints.get(nom_endpoint)
print("Trafic :", endpoint.traffic)
print("URL :", endpoint.scoring_uri)

Trafic : {'blue': 100}
URL : https://students-ep-arthur-louca-1904.germanywestcentral.inference.ml.azure.com/score


In [9]:
reponse = ml_client.online_endpoints.invoke(endpoint_name=nom_endpoint,
                                            request_file="sample-request.json")
print(reponse)

{"predictions": [0, 0, 1], "probabilites_abandon": [0.461, 0.5, 0.998]}


In [10]:
cle = ml_client.online_endpoints.get_keys(nom_endpoint).primary_key
entetes = {"Content-Type": "application/json",
           "Authorization": f"Bearer {cle}"}

r = requests.post(endpoint.scoring_uri, data=json.dumps(requete), headers=entetes)
print("Code HTTP :", r.status_code)
print("Réponse   :", r.json())

Code HTTP : 200
Réponse   : {'predictions': [0, 0, 1], 'probabilites_abandon': [0.461, 0.5, 0.998]}


In [11]:
requete_fausse = {"data": [exemples.iloc[0, :1].to_dict()]}
r = requests.post(endpoint.scoring_uri, data=json.dumps(requete_fausse), headers=entetes)
print(r.json())

{'erreur': '35 colonne(s) manquante(s) : [\'Application mode\', \'Application order\', \'Course\', \'Daytime/evening attendance\', \'Previous qualification\', \'Previous qualification (grade)\', \'Nacionality\', "Mother\'s qualification", "Father\'s qualification", "Mother\'s occupation", "Father\'s occupation", \'Admission grade\', \'Displaced\', \'Educational special needs\', \'Debtor\', \'Tuition fees up to date\', \'Gender\', \'Scholarship holder\', \'Age at enrollment\', \'International\', \'Curricular units 1st sem (credited)\', \'Curricular units 1st sem (enrolled)\', \'Curricular units 1st sem (evaluations)\', \'Curricular units 1st sem (approved)\', \'Curricular units 1st sem (grade)\', \'Curricular units 1st sem (without evaluations)\', \'Curricular units 2nd sem (credited)\', \'Curricular units 2nd sem (enrolled)\', \'Curricular units 2nd sem (evaluations)\', \'Curricular units 2nd sem (approved)\', \'Curricular units 2nd sem (grade)\', \'Curricular units 2nd sem (without ev